In [56]:
"""
Modular RAG.

Unlike Naive RAG (fixed retrieval pipeline) and Advanced RAG (improved retrieval techniques such as query rewriting and reranking), Modular RAG uses interchangeable components that can be independently configured, combined and orchestrated depending on the problem.

We'll explore multiple retrieval strategies, routing, query decomposition and other modules, using your existing Ollama + FAISS setup.


We added a 2nd source here "human_nutrition_and_dietary_requirement.txt"
"""

'\nModular RAG.\n\nUnlike Naive RAG (fixed retrieval pipeline) and Advanced RAG (improved retrieval techniques such as query rewriting and reranking), Modular RAG uses interchangeable components that can be independently configured, combined and orchestrated depending on the problem.\n\nWe\'ll explore multiple retrieval strategies, routing, query decomposition and other modules, using your existing Ollama + FAISS setup.\n\n\nWe added a 2nd source here "human_nutrition_and_dietary_requirement.txt"\n'

In [57]:
"""
Modular RAG — Proposed flow
1. Document Loading: TextLoader loads fruits_and_vegetables.txt.

2. Text Splitting: RecursiveCharacterTextSplitter divides the document into chunks.

3. Embedding Generation: OllamaEmbeddings converts chunks into vectors.

4. Vector Storage: FAISS stores the vectors and their corresponding Documents.

5. Query Rewriting Module: Qwen rewrites the user's original query to improve retrieval.

6. Query Routing Module: Determines which retrieval strategy or document collection should handle the query. Initially, we'll keep just one route: our existing FAISS vector store.

7. Retrieval Module: Fetches the top relevant chunks from FAISS.

8. Context Preparation Module: Prepares the retrieved chunks for the final prompt.

9. Prompt Module: Combines the retrieved context with the original user question.

10. Generation Module: Qwen generates the answer using the supplied context.

11. Output Parsing Module: StrOutputParser() extracts the final response as a string.

"""

"\nModular RAG — Proposed flow\n1. Document Loading: TextLoader loads fruits_and_vegetables.txt.\n\n2. Text Splitting: RecursiveCharacterTextSplitter divides the document into chunks.\n\n3. Embedding Generation: OllamaEmbeddings converts chunks into vectors.\n\n4. Vector Storage: FAISS stores the vectors and their corresponding Documents.\n\n5. Query Rewriting Module: Qwen rewrites the user's original query to improve retrieval.\n\n6. Query Routing Module: Determines which retrieval strategy or document collection should handle the query. Initially, we'll keep just one route: our existing FAISS vector store.\n\n7. Retrieval Module: Fetches the top relevant chunks from FAISS.\n\n8. Context Preparation Module: Prepares the retrieved chunks for the final prompt.\n\n9. Prompt Module: Combines the retrieved context with the original user question.\n\n10. Generation Module: Qwen generates the answer using the supplied context.\n\n11. Output Parsing Module: StrOutputParser() extracts the fina

In [58]:
from langchain_community.document_loaders import TextLoader
# Document Loading:
loader_fruits = TextLoader("./notes/fruits_and_vegetables.txt")

documents_fruits = loader_fruits.load()

loader_diet = TextLoader("./notes/human_nutrition_and_dietary_requirements.txt")

documents_diet = loader_diet.load()

In [59]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
#  Text Splitting:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=1
)

chunks_fruits = text_splitter.split_documents(documents_fruits)
chunks_diet = text_splitter.split_documents(documents_diet)

In [60]:
from langchain_community.vectorstores import FAISS
from langchain_ollama import OllamaEmbeddings

#Embedding Generation, Vector Storage
embedding_model = OllamaEmbeddings(
    model="nomic-embed-text"
)
vectorstore_fruits = FAISS.from_documents(chunks_fruits, embedding_model)
vectorstore_fruits.save_local("./faiss_fruits")

vectorstore_diet = FAISS.from_documents(chunks_diet, embedding_model)
vectorstore_diet.save_local("./faiss_diet")

################################################################

# Retrieve both fruits and diet— load back and search
vectorstore_fruits = FAISS.load_local(
    "./faiss_fruits",
    embedding_model,
    allow_dangerous_deserialization=True
)
vectorstore_diet = FAISS.load_local(
    "./faiss_diet",
    embedding_model,
    allow_dangerous_deserialization=True
)


fruits_retriever = vectorstore_fruits.as_retriever(
    search_kwargs={"k": 2}
)
nutrition_retriever = vectorstore_diet.as_retriever(
    search_kwargs={"k": 2}
)

In [61]:
from langchain_ollama import OllamaLLM

llm = OllamaLLM(model="qwen2.5:1.5b", temperature=0.2)

In [62]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda

# Step 1: Define the Query Router prompt
router_prompt = ChatPromptTemplate.from_template("""
        You are a query router.

        Choose the most appropriate knowledge source for the user's question.

        Available sources:

        fruits:
        Information about fruits, vegetables, colours, seeds, plant parts,
        taste and cooking.

        nutrition:
        Information about vitamins, minerals, proteins, carbohydrates,
        fats, fibre, hydration and dietary requirements.

        Return ONLY one word: fruits or nutrition.

        User question: {question}
""")

# Step 2: Create the Query Router
query_router = router_prompt | llm | StrOutputParser()


query_rewrite_prompt = ChatPromptTemplate.from_template(
    """
Rewrite the user's question into a clearer search query
for retrieving relevant information from documents.

Do not answer the question.
Do not change its original intent.
Return only the rewritten query.

User question: {question}
    """
)
query_rewriter = query_rewrite_prompt | llm | StrOutputParser()


"""

You've understood the important part. route_and_retrieve() is our custom-defined method that controls the flow: it decides which knowledge source to use, retrieves the documents, and passes the resulting context to the LLM.

One small distinction: the router LLM decides the route; our Python method orchestrates the execution.

Futher we will pass this route_and_retrieve into RunnableLamda
it has a very Java-like feel when it comes to its name! 
RunnableLambda in LCEL takes a normal Python function and converts it into a Runnable, so that LangChain can execute it as part of an LCEL pipeline using |. In our example, route_and_retrieve is an ordinary Python function, but RunnableLambda(route_and_retrieve) makes it compatible with LCEL. Think of it like adapting a regular Java method into something that can participate in a processing pipeline. The function still does exactly what we wrote; RunnableLambda simply gives it the LCEL interface, including .invoke().

In short: Normal Python function → RunnableLambda → LCEL-compatible Runnable.

"""

# Step 3: Route the question and retrieve relevant documents
def route_and_retrieve(question):

    # Ask the LLM which knowledge source to use
    selected_route = query_router.invoke(
        {"question": question}
    ).strip().lower()

    print("Selected Route:", selected_route)

    # Select the appropriate retriever
    if selected_route == "fruits":
        selected_retriever = fruits_retriever

    elif selected_route == "nutrition":
        selected_retriever = nutrition_retriever

    else:
        raise ValueError("Invalid route selected: " + selected_route)

    # Rewrite the original question for better retrieval
    rewritten_query = query_rewriter.invoke(
        {"question": question}
    )

    print("Rewritten Query:", rewritten_query)

    # Retrieve documents from the selected knowledge source AUGMENTATION DONE
    retrieved_docs = selected_retriever.invoke(rewritten_query)

    # Combine retrieved document content into context
    context = "\n\n".join(
        doc.page_content for doc in retrieved_docs
    )

    # Return both context and original question
    return {
        "context": context,
        "question": question
    }


prompt = ChatPromptTemplate.from_template("""
Answer the question using the following context.

Context:
{context}

Question:
{question}
""")

# Step 4: Build the final RAG chain
chain = (
    RunnableLambda(route_and_retrieve)
    | prompt
    | llm
    | StrOutputParser()
)


# Step 5: Test
response = chain.invoke(
    "What is carbohydrates? Do they give hydration?"
)

print("Final response is : "+ response)

Selected Route: nutrition
Rewritten Query: Carbohydrates: Definition and Hydration Role
Final response is : Carbohydrates are an important source of energy. They are found in foods like rice, wheat, potatoes, oats, and fruits. The body breaks down carbohydrates into glucose, which can be used by cells for energy. While water is essential for hydration, carbohydrates do not provide hydration.
